# Seismology basics with ObsPy

ObsPy is the standard Python toolkit for working with seismic waveform data. We use its
bundled example seismogram (a real 3-component recording, no download needed) and run the
standard first processing steps: inspect the raw trace, detrend it, and bandpass-filter it to
the frequency band typical of local earthquakes.

In [ ]:
import obspy

# obspy.read() with no arguments loads ObsPy's bundled example stream - a real
# 3-component (Z/N/E) seismogram from station BW.RJOB.
st = obspy.read()
print(f"{len(st)} trace(s):")
for tr in st:
    print(f"  {tr.id}: {tr.stats.npts} samples @ {tr.stats.sampling_rate} Hz, "
          f"starts {tr.stats.starttime}")

In [ ]:
st.plot();

## Standard preprocessing: detrend and filter

Raw seismic traces usually have an instrument offset (a nonzero mean/trend) and contain noise
outside the frequency band you care about. `detrend` removes the offset; a bandpass filter
keeps only the 1-10 Hz band, typical for local earthquake signals.

In [ ]:
tr = st[0].copy()
print(f"raw:               mean={tr.data.mean():.2f}")

tr.detrend("linear")
tr.filter("bandpass", freqmin=1.0, freqmax=10.0, corners=4, zerophase=True)
print(f"detrend+bandpass:  mean={tr.data.mean():.4f}  (should be ~0 after detrending)")

tr.plot();

## A basic feature: peak amplitude and arrival time

The simplest possible "where's the interesting part of this signal" feature - the sample with
the largest absolute amplitude, and how far into the trace it occurs. Real earthquake analysis
builds on exactly this kind of feature (amplitude, timing) for magnitude estimation and phase
picking, just with much more sophisticated methods.

In [ ]:
import numpy as np

peak_idx = np.argmax(np.abs(tr.data))
peak_time = peak_idx / tr.stats.sampling_rate
peak_amp = np.abs(tr.data).max()
print(f"peak absolute amplitude: {peak_amp:.1f}, at t={peak_time:.2f}s into the trace")

## Next steps

- Try different filter bands (`freqmin`/`freqmax`) and see how the waveform shape changes.
- Load real data instead of the bundled example: ObsPy can fetch waveforms directly from
  public data centers via `obspy.clients.fdsn.Client` (needs internet access - the bundled
  example above deliberately doesn't).
- Compute a spectrogram (`tr.spectrogram()`) to see how the signal's frequency content changes
  over time, rather than just its time-domain amplitude.